# 09 — Retrieval-Augmented Generation (RAG) Maintenance Layer

## Purpose

This notebook adds the **RAG layer** to the Smart Manufacturing AI system.

The Memory layer answers:

> **What happened to this machine previously?**

RAG answers:

> **What does the available maintenance knowledge say about this machine, fault, and situation?**

The final RAG pipeline combines:

```text
Current machine event
        +
Machine historical memory
        +
Relevant maintenance knowledge
        ↓
       LLM
        ↓
Context-aware maintenance intelligence
```

### Important design decision

The RAG knowledge base used here is a **project knowledge base created specifically for the simulation**. It contains structured maintenance guidance for the simulated machine/fault scenarios.

This is intentionally separate from machine Memory:

- **Memory** = historical observations generated by our own Smart Twin/agent pipeline.
- **Knowledge Base** = maintenance/domain guidance that can be retrieved when needed.
- **LLM** = language reasoning and explanation layer.

The RAG retriever in this notebook uses **TF-IDF similarity** from scikit-learn. This keeps the project lightweight and reproducible while still implementing the complete retrieval → augmentation → generation workflow.

The LLM does not replace the ML models, agents, or orchestrator. Their decisions remain authoritative; the RAG-LLM layer explains those decisions using historical memory and retrieved maintenance knowledge.


In [1]:
# ============================================
# STEP 1: IMPORTS AND PROJECT PATHS
# ============================================

from pathlib import Path
import os
import json
import re

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("RAG environment ready.")
print("Current working directory:", Path.cwd())


RAG environment ready.
Current working directory: /home/labuser/Desktop/Project/3203141-Capstone_Proj/notebooks


In [2]:
# ============================================
# STEP 2: LOCATE PROJECT ARTIFACTS
# ============================================

possible_roots = [
    Path.cwd(),
    Path.cwd().parent
]

processed_dir = None

for root in possible_roots:
    candidate = root / "data" / "processed"
    if candidate.exists():
        processed_dir = candidate
        break

if processed_dir is None:
    raise FileNotFoundError(
        "Could not locate data/processed. "
        "Run the previous project notebooks first."
    )

memory_path = processed_dir / "machine_memory.csv"
summary_path = processed_dir / "machine_memory_summary.csv"
llm_reports_path = processed_dir / "llm_maintenance_reports.csv"

if not memory_path.exists():
    raise FileNotFoundError(
        f"{memory_path} was not found. "
        "Complete 08_Memory.ipynb first."
    )

print("Processed directory:", processed_dir)
print("Machine memory:", memory_path)
print("Machine summary:", summary_path if summary_path.exists() else "Not found")
print("LLM reports:", llm_reports_path if llm_reports_path.exists() else "Not found")


Processed directory: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed
Machine memory: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/machine_memory.csv
Machine summary: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/machine_memory_summary.csv
LLM reports: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/llm_maintenance_reports.csv


In [3]:
# ============================================
# STEP 3: LOAD MACHINE MEMORY
# ============================================

memory_df = pd.read_csv(memory_path)

print("Memory shape:", memory_df.shape)
print("Machines:", memory_df["Machine_ID"].nunique())

display(
    memory_df[
        [
            "Machine_ID",
            "Machine_Type",
            "Cycle",
            "Simulation_Fault",
            "failure_probability",
            "predicted_rul_cycles",
            "overall_status",
            "maintenance_priority",
            "maintenance_decision",
            "alert_level"
        ]
    ].head(5)
)


Memory shape: (55, 28)
Machines: 7


,Machine_ID,Machine_Type,Cycle,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,maintenance_priority,maintenance_decision,alert_level
0,CNC_001,CNC_Mill,193,BEARING_WEAR,0.0002,47.87,WARNING,HIGH,SCHEDULE INSPECTION,WARNING
1,CNC_001,CNC_Mill,195,BEARING_WEAR,0.0002,49.58,WARNING,HIGH,SCHEDULE INSPECTION,WARNING
2,CNC_001,CNC_Mill,221,BEARING_WEAR,0.0003,17.02,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL
3,CNC_001,CNC_Mill,223,BEARING_WEAR,0.7748,13.21,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL
4,CNC_001,CNC_Mill,224,BEARING_WEAR,0.4115,13.86,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL


In [4]:
# ============================================
# STEP 4: CREATE SIMULATION MAINTENANCE
#         KNOWLEDGE BASE
# ============================================

# These documents represent domain/maintenance guidance
# for the simulated fault scenarios used by this project.
# They are intentionally kept as separate documents so the
# retriever can select only the relevant knowledge.

knowledge_documents = [
    {
        "document_id": "KB_ACTUATOR_001",
        "title": "Industrial Robot — Actuator Degradation",
        "machine_type": "Industrial_Robot",
        "fault": "ACTUATOR_DEGRADATION",
        "content": (
            "Actuator degradation in an industrial robot can reduce "
            "positioning accuracy, responsiveness, and repeatability. "
            "Inspect actuator assemblies for abnormal wear, backlash, "
            "unusual noise, overheating, or reduced response. Verify "
            "motor current, feedback signals, gear or transmission "
            "condition, and controller alarms. If degradation is "
            "progressing, schedule an inspection before production "
            "quality is affected. If severe loss of control or unsafe "
            "motion is detected, stop the machine according to site "
            "safety procedures and escalate to maintenance."
        )
    },
    {
        "document_id": "KB_BEARING_001",
        "title": "Rotating Equipment — Bearing Wear",
        "machine_type": "General_Rotating_Equipment",
        "fault": "BEARING_WEAR",
        "content": (
            "Bearing wear commonly appears as increasing vibration, "
            "temperature rise, acoustic changes, or abnormal rotational "
            "behavior. Inspect bearing condition, lubrication, shaft "
            "alignment, and mounting. Compare vibration and temperature "
            "against normal operating ranges. Increasing vibration with "
            "declining RUL should increase maintenance urgency. Severe "
            "temperature or vibration conditions require immediate "
            "operator attention and maintenance escalation."
        )
    },
    {
        "document_id": "KB_THERMAL_001",
        "title": "Industrial Equipment — Thermal Overload",
        "machine_type": "General_Industrial_Machine",
        "fault": "THERMAL_OVERLOAD",
        "content": (
            "Thermal overload indicates that equipment temperature is "
            "above its expected operating condition. Check cooling "
            "systems, airflow, heat exchangers, fans, coolant levels, "
            "ambient conditions, and excessive mechanical load. Confirm "
            "temperature sensor validity before maintenance decisions. "
            "Persistent overheating can accelerate component degradation. "
            "If temperature continues to rise toward unsafe limits, "
            "follow shutdown and escalation procedures."
        )
    },
    {
        "document_id": "KB_PRESSURE_001",
        "title": "Process Equipment — Pressure Anomaly",
        "machine_type": "General_Process_Equipment",
        "fault": "PRESSURE_ANOMALY",
        "content": (
            "Pressure anomalies can result from blocked lines, leaks, "
            "valve problems, pump performance changes, sensor faults, "
            "or abnormal process conditions. Verify pressure readings "
            "against redundant or reference measurements where available. "
            "Inspect valves, seals, filters, pipes, and connected process "
            "equipment. A rapidly worsening pressure anomaly should be "
            "escalated because it may affect safe operation."
        )
    },
    {
        "document_id": "KB_ELECTRICAL_001",
        "title": "Industrial Equipment — Electrical Overload",
        "machine_type": "General_Industrial_Machine",
        "fault": "ELECTRICAL_OVERLOAD",
        "content": (
            "Electrical overload may be associated with excessive current, "
            "motor loading, insulation problems, supply issues, or "
            "mechanical resistance. Inspect current draw, motor load, "
            "connections, protection devices, and relevant electrical "
            "components. Check whether current increases consistently "
            "with load or appears unexpectedly. Persistent overload "
            "should be investigated promptly; abnormal electrical "
            "conditions should be handled by qualified personnel."
        )
    },
    {
        "document_id": "KB_VIBRATION_001",
        "title": "Industrial Equipment — Excessive Vibration",
        "machine_type": "General_Rotating_Equipment",
        "fault": "VIBRATION_ANOMALY",
        "content": (
            "Excessive vibration can indicate imbalance, misalignment, "
            "looseness, bearing degradation, coupling problems, or "
            "structural issues. Compare vibration against the machine's "
            "normal baseline and inspect mechanical connections, "
            "alignment, bearings, and rotating components. Increasing "
            "vibration combined with falling RUL is a strong reason to "
            "prioritize inspection."
        )
    },
    {
        "document_id": "KB_LUBRICATION_001",
        "title": "Industrial Equipment — Lubrication Degradation",
        "machine_type": "General_Industrial_Machine",
        "fault": "LUBRICATION_DEGRADATION",
        "content": (
            "Lubrication degradation can increase friction, heat, wear, "
            "and vibration. Check lubricant level, contamination, "
            "viscosity condition, leaks, and lubrication intervals. "
            "Inspect components for abnormal wear or temperature. "
            "Correct lubrication issues according to the equipment's "
            "maintenance procedure and verify that the condition "
            "stabilizes after intervention."
        )
    },
    {
        "document_id": "KB_GENERAL_001",
        "title": "Predictive Maintenance — Warning and Critical Conditions",
        "machine_type": "General",
        "fault": "GENERAL",
        "content": (
            "A WARNING condition indicates that the machine is showing "
            "a condition requiring increased monitoring or planned "
            "inspection. A CRITICAL condition indicates substantially "
            "higher operational risk and should receive immediate "
            "maintenance attention according to the system decision. "
            "Failure probability and RUL should be interpreted together "
            "with machine status, fault type, trend information, and "
            "agent decisions. The predictive system is decision support; "
            "physical inspection and site safety procedures remain "
            "authoritative."
        )
    }
]

knowledge_df = pd.DataFrame(knowledge_documents)

print("Knowledge documents:", len(knowledge_df))
display(
    knowledge_df[
        ["document_id", "title", "machine_type", "fault"]
    ]
)


Knowledge documents: 8


,document_id,title,machine_type,fault
0,KB_ACTUATOR_001,Industrial Robot — Actuator Degradation,Industrial_Robot,ACTUATOR_DEGRADATION
1,KB_BEARING_001,Rotating Equipment — Bearing Wear,General_Rotating_Equipment,BEARING_WEAR
2,KB_THERMAL_001,Industrial Equipment — Thermal Overload,General_Industrial_Machine,THERMAL_OVERLOAD
3,KB_PRESSURE_001,Process Equipment — Pressure Anomaly,General_Process_Equipment,PRESSURE_ANOMALY
4,KB_ELECTRICAL_001,Industrial Equipment — Electrical Overload,General_Industrial_Machine,ELECTRICAL_OVERLOAD
5,KB_VIBRATION_001,Industrial Equipment — Excessive Vibration,General_Rotating_Equipment,VIBRATION_ANOMALY
6,KB_LUBRICATION_001,Industrial Equipment — Lubrication Degradation,General_Industrial_Machine,LUBRICATION_DEGRADATION
7,KB_GENERAL_001,Predictive Maintenance — Warning and Critical ...,General,GENERAL


In [5]:
# ============================================
# STEP 5: SAVE KNOWLEDGE BASE
# ============================================

knowledge_dir = processed_dir.parent / "knowledge"
knowledge_dir.mkdir(parents=True, exist_ok=True)

knowledge_path = knowledge_dir / "maintenance_knowledge_base.json"

with open(knowledge_path, "w", encoding="utf-8") as f:
    json.dump(
        knowledge_documents,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Knowledge base saved:")
print(knowledge_path)


Knowledge base saved:
/home/labuser/Desktop/Project/3203141-Capstone_Proj/data/knowledge/maintenance_knowledge_base.json


In [6]:
# ============================================
# STEP 6: PREPARE RETRIEVAL TEXT
# ============================================

def build_document_text(doc):
    return " ".join([
        doc["title"],
        doc["machine_type"],
        doc["fault"],
        doc["content"]
    ])

knowledge_df["retrieval_text"] = knowledge_df.apply(
    build_document_text,
    axis=1
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

document_matrix = vectorizer.fit_transform(
    knowledge_df["retrieval_text"]
)

print("TF-IDF retrieval index created.")
print("Documents indexed:", document_matrix.shape[0])
print("Vocabulary size:", len(vectorizer.vocabulary_))


TF-IDF retrieval index created.
Documents indexed: 8
Vocabulary size: 567


In [7]:
# ============================================
# STEP 7: RAG RETRIEVAL FUNCTION
# ============================================

def retrieve_knowledge(
    query,
    top_k=3
):
    """
    Retrieve the most relevant maintenance documents
    using TF-IDF cosine similarity.
    """

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        document_matrix
    ).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = knowledge_df.iloc[top_indices].copy()
    results["similarity_score"] = scores[top_indices]

    return (
        results
        .sort_values("similarity_score", ascending=False)
        .reset_index(drop=True)
    )


print("RAG retrieval function created.")


RAG retrieval function created.


In [8]:
# ============================================
# STEP 8: TEST RETRIEVAL
# ============================================

test_query = (
    "industrial robot actuator degradation "
    "warning low RUL inspection maintenance"
)

retrieved = retrieve_knowledge(
    test_query,
    top_k=3
)

print("Query:")
print(test_query)

print("\nRetrieved knowledge:")
display(
    retrieved[
        [
            "document_id",
            "title",
            "fault",
            "similarity_score"
        ]
    ]
)

print("\nTop document content:")
print(retrieved.iloc[0]["content"])


Query:
industrial robot actuator degradation warning low RUL inspection maintenance

Retrieved knowledge:


,document_id,title,fault,similarity_score
0,KB_ACTUATOR_001,Industrial Robot — Actuator Degradation,ACTUATOR_DEGRADATION,0.413617
1,KB_GENERAL_001,Predictive Maintenance — Warning and Critical ...,GENERAL,0.139050
2,KB_VIBRATION_001,Industrial Equipment — Excessive Vibration,VIBRATION_ANOMALY,0.072227



Top document content:
Actuator degradation in an industrial robot can reduce positioning accuracy, responsiveness, and repeatability. Inspect actuator assemblies for abnormal wear, backlash, unusual noise, overheating, or reduced response. Verify motor current, feedback signals, gear or transmission condition, and controller alarms. If degradation is progressing, schedule an inspection before production quality is affected. If severe loss of control or unsafe motion is detected, stop the machine according to site safety procedures and escalate to maintenance.


In [9]:
# ============================================
# STEP 9: BUILD MEMORY CONTEXT
# ============================================

def retrieve_machine_memory(
    machine_id,
    current_cycle=None,
    top_k=5
):
    history = memory_df[
        memory_df["Machine_ID"] == machine_id
    ].copy()

    if current_cycle is not None:
        history = history[
            history["Cycle"] < current_cycle
        ]

    return (
        history
        .sort_values("Cycle", ascending=False)
        .head(top_k)
        .sort_values("Cycle")
        .reset_index(drop=True)
    )


def build_memory_context(
    machine_id,
    current_cycle=None,
    top_k=5
):
    history = retrieve_machine_memory(
        machine_id=machine_id,
        current_cycle=current_cycle,
        top_k=top_k
    )

    if history.empty:
        return "No previous machine history was found."

    lines = [
        f"Historical memory for {machine_id}:"
    ]

    for _, row in history.iterrows():
        lines.append(
            f"- Cycle {int(row['Cycle'])}: "
            f"status={row['overall_status']}; "
            f"failure_probability="
            f"{row['failure_probability']:.4f}; "
            f"RUL={row['predicted_rul_cycles']:.2f}; "
            f"fault={row['Simulation_Fault']}; "
            f"maintenance={row['maintenance_decision']}; "
            f"alert={row['alert_level']}."
        )

    return "\n".join(lines)


print("Memory retrieval is ready for RAG.")


Memory retrieval is ready for RAG.


In [10]:
# ============================================
# STEP 10: BUILD RAG QUERY FROM A MACHINE EVENT
# ============================================

def build_rag_query(row):
    return (
        f"machine type {row['Machine_Type']} "
        f"fault {row['Simulation_Fault']} "
        f"status {row['overall_status']} "
        f"maintenance priority {row['maintenance_priority']} "
        f"maintenance decision {row['maintenance_decision']} "
        f"alert {row['alert_level']} "
        f"RUL {row['predicted_rul_cycles']} "
        f"failure probability {row['failure_probability']}"
    )


sample_event = memory_df.iloc[0]

sample_query = build_rag_query(sample_event)
sample_retrieved = retrieve_knowledge(
    sample_query,
    top_k=3
)

print("RAG query:")
print(sample_query)

print("\nRetrieved documents:")
display(
    sample_retrieved[
        [
            "document_id",
            "title",
            "fault",
            "similarity_score"
        ]
    ]
)


RAG query:
machine type CNC_Mill fault BEARING_WEAR status WARNING maintenance priority HIGH maintenance decision SCHEDULE INSPECTION alert WARNING RUL 47.87 failure probability 0.0002

Retrieved documents:


,document_id,title,fault,similarity_score
0,KB_GENERAL_001,Predictive Maintenance — Warning and Critical ...,GENERAL,0.364217
1,KB_ACTUATOR_001,Industrial Robot — Actuator Degradation,ACTUATOR_DEGRADATION,0.083116
2,KB_BEARING_001,Rotating Equipment — Bearing Wear,BEARING_WEAR,0.066986


In [11]:
# ============================================
# STEP 11: BUILD AUGMENTED CONTEXT
# ============================================

def build_augmented_context(
    row,
    memory_top_k=5,
    knowledge_top_k=3
):
    # Current machine event
    current_event = (
        f"Machine ID: {row['Machine_ID']}\n"
        f"Machine Type: {row['Machine_Type']}\n"
        f"Cycle: {row['Cycle']}\n"
        f"Simulation Fault: {row['Simulation_Fault']}\n"
        f"Failure Probability: {row['failure_probability']}\n"
        f"Predicted RUL: {row['predicted_rul_cycles']} cycles\n"
        f"Overall Status: {row['overall_status']}\n"
        f"Maintenance Priority: {row['maintenance_priority']}\n"
        f"Maintenance Decision: {row['maintenance_decision']}\n"
        f"Alert Level: {row['alert_level']}\n"
        f"Alert Required: {row['alert_required']}"
    )

    # Historical memory
    memory_context = build_memory_context(
        machine_id=row["Machine_ID"],
        current_cycle=row["Cycle"],
        top_k=memory_top_k
    )

    # Retrieved knowledge
    query = build_rag_query(row)

    retrieved = retrieve_knowledge(
        query,
        top_k=knowledge_top_k
    )

    knowledge_lines = [
        "Retrieved maintenance knowledge:"
    ]

    for _, doc in retrieved.iterrows():
        knowledge_lines.append(
            f"[{doc['document_id']}] {doc['title']} "
            f"(similarity={doc['similarity_score']:.3f})\n"
            f"{doc['content']}"
        )

    knowledge_context = "\n".join(knowledge_lines)

    return (
        "CURRENT MACHINE EVENT\n"
        "=====================\n"
        f"{current_event}\n\n"
        "MACHINE MEMORY\n"
        "===============\n"
        f"{memory_context}\n\n"
        "RETRIEVED MAINTENANCE KNOWLEDGE\n"
        "================================\n"
        f"{knowledge_context}"
    )


sample_augmented_context = build_augmented_context(
    sample_event
)

print(sample_augmented_context)


CURRENT MACHINE EVENT
Machine ID: CNC_001
Machine Type: CNC_Mill
Cycle: 193
Simulation Fault: BEARING_WEAR
Failure Probability: 0.0002
Predicted RUL: 47.87 cycles
Overall Status: WARNING
Maintenance Priority: HIGH
Maintenance Decision: SCHEDULE INSPECTION
Alert Level: WARNING
Alert Required: True

MACHINE MEMORY
No previous machine history was found.

RETRIEVED MAINTENANCE KNOWLEDGE
Retrieved maintenance knowledge:
[KB_GENERAL_001] Predictive Maintenance — Warning and Critical Conditions (similarity=0.364)
A WARNING condition indicates that the machine is showing a condition requiring increased monitoring or planned inspection. A CRITICAL condition indicates substantially higher operational risk and should receive immediate maintenance attention according to the system decision. Failure probability and RUL should be interpreted together with machine status, fault type, trend information, and agent decisions. The predictive system is decision support; physical inspection and site safety

In [12]:
# ============================================
# STEP 12: CONNECT TO TCS MAAS LLM
# ============================================

from langchain_openai import ChatOpenAI
import httpx

api_key = os.getenv("GENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GENAI_API_KEY is not available in this Jupyter kernel. "
        "Start Jupyter from the terminal where the API key was exported."
    )

client = httpx.Client(verify=False)

llm = ChatOpenAI(
    base_url="https://genailab.tcs.in",
    model="genailab-maas-gpt-35-turbo",
    api_key=api_key,
    http_client=client
)

print("MaaS LLM connection object created successfully.")


MaaS LLM connection object created successfully.


In [13]:
# ============================================
# STEP 13: RAG-ENABLED MAINTENANCE AGENT
# ============================================

RAG_SYSTEM_PROMPT = """
You are the Retrieval-Augmented Maintenance Intelligence Agent
for a Smart Manufacturing Digital Twin system.

Your task is to explain an existing machine condition using:
1. the current machine event,
2. historical machine memory,
3. retrieved maintenance knowledge.

IMPORTANT RULES:
- Do not replace or contradict the Failure Prediction, RUL,
  Maintenance Agent, Alert Agent, or Orchestrator decisions.
- Treat those system decisions as authoritative.
- Retrieved knowledge is supporting domain guidance, not a replacement
  for physical inspection, safety procedures, or qualified personnel.
- Clearly distinguish observed/model-predicted evidence from possible causes.
- Do not invent sensor readings or maintenance history.
- Keep recommendations practical and traceable to the supplied context.

Use this structure:

1. CURRENT ASSESSMENT
2. HISTORICAL CONTEXT
3. EVIDENCE
4. RELEVANT MAINTENANCE KNOWLEDGE
5. RECOMMENDED INSPECTION
6. OPERATOR / MAINTENANCE ACTION
7. ESCALATION CONDITIONS
"""


def rag_maintenance_agent(context):
    response = llm.invoke(
        [
            ("system", RAG_SYSTEM_PROMPT),
            (
                "human",
                "Analyze the following machine event using the supplied "
                "memory and retrieved maintenance knowledge.\n\n"
                + context
            )
        ]
    )

    return response.content


print("RAG-enabled maintenance agent defined.")


RAG-enabled maintenance agent defined.


In [14]:
# ============================================
# STEP 14: TEST ONE RAG-GROUNDED REPORT
# ============================================

test_context = build_augmented_context(
    sample_event,
    memory_top_k=5,
    knowledge_top_k=3
)

test_rag_report = rag_maintenance_agent(
    test_context
)

print("============================================")
print("RAG-GROUNDED MAINTENANCE REPORT")
print("============================================")
print(test_rag_report)


RAG-GROUNDED MAINTENANCE REPORT
1. CURRENT ASSESSMENT  
The machine CNC_001, a CNC Mill, is currently in a WARNING state due to a simulation fault indicating BEARING_WEAR. The failure probability is low (0.0002) but the predicted Remaining Useful Life (RUL) is approximately 47.87 cycles, signaling an approaching maintenance need. The maintenance priority is HIGH and the system has decided to SCHEDULE INSPECTION. An alert has been issued accordingly.

2. HISTORICAL CONTEXT  
There is no previous machine history available for CNC_001. This event is the first recorded indication of a bearing wear fault.

3. EVIDENCE  
- Simulation fault specifically identifies bearing wear.  
- Failure probability is low but nonzero, indicating early-stage degradation.  
- Predicted RUL under 50 cycles suggests that bearing condition might worsen soon.  
- Overall status is WARNING, consistent with an initial sign of deterioration needing monitoring or planned inspection.

4. RELEVANT MAINTENANCE KNOWLEDG

In [15]:
# ============================================
# STEP 15: GENERATE RAG REPORTS FOR ALL
#          ACTIONABLE EVENTS
# ============================================

rag_reports = []

for index, row in memory_df.iterrows():

    try:
        context = build_augmented_context(
            row,
            memory_top_k=5,
            knowledge_top_k=3
        )

        report = rag_maintenance_agent(
            context
        )

        retrieved_docs = retrieve_knowledge(
            build_rag_query(row),
            top_k=3
        )

        rag_reports.append({
            "Machine_ID": row["Machine_ID"],
            "Machine_Type": row["Machine_Type"],
            "Cycle": row["Cycle"],
            "Simulation_Fault": row["Simulation_Fault"],
            "failure_probability": row["failure_probability"],
            "predicted_rul_cycles": row["predicted_rul_cycles"],
            "overall_status": row["overall_status"],
            "maintenance_priority": row["maintenance_priority"],
            "maintenance_decision": row["maintenance_decision"],
            "alert_level": row["alert_level"],
            "alert_required": row["alert_required"],
            "retrieved_document_ids": ", ".join(
                retrieved_docs["document_id"].tolist()
            ),
            "retrieved_document_titles": " | ".join(
                retrieved_docs["title"].tolist()
            ),
            "rag_report": report
        })

        print(
            f"Processed {len(rag_reports)}/{len(memory_df)} "
            f"- {row['Machine_ID']} Cycle {row['Cycle']}"
        )

    except Exception as e:
        print(
            f"ERROR processing row {index}: {e}"
        )

print(
    f"Completed RAG generation: "
    f"{len(rag_reports)}/{len(memory_df)}"
)


Processed 1/55 - CNC_001 Cycle 193
Processed 2/55 - CNC_001 Cycle 195
Processed 3/55 - CNC_001 Cycle 221
Processed 4/55 - CNC_001 Cycle 223
Processed 5/55 - CNC_001 Cycle 224
Processed 6/55 - CNC_001 Cycle 225
Processed 7/55 - CNC_001 Cycle 229
Processed 8/55 - COMP_003 Cycle 218
Processed 9/55 - COMP_003 Cycle 222
Processed 10/55 - COMP_003 Cycle 224
Processed 11/55 - COMP_003 Cycle 247
Processed 12/55 - COMP_003 Cycle 248
Processed 13/55 - COMP_003 Cycle 251
Processed 14/55 - COMP_003 Cycle 253
Processed 15/55 - COMP_003 Cycle 254
Processed 16/55 - COMP_003 Cycle 255
Processed 17/55 - MOTOR_004 Cycle 173
Processed 18/55 - MOTOR_004 Cycle 178
Processed 19/55 - MOTOR_004 Cycle 180
Processed 20/55 - MOTOR_004 Cycle 208
Processed 21/55 - MOTOR_004 Cycle 210
Processed 22/55 - MOTOR_004 Cycle 211
Processed 23/55 - MOTOR_004 Cycle 212
Processed 24/55 - MOTOR_004 Cycle 216
Processed 25/55 - MOTOR_004 Cycle 217
Processed 26/55 - PRESS_006 Cycle 202
Processed 27/55 - PRESS_006 Cycle 206
Proces

In [16]:
# ============================================
# STEP 16: CREATE RAG REPORT DATAFRAME
# ============================================

rag_reports_df = pd.DataFrame(rag_reports)

print("RAG reports:", len(rag_reports_df))
print("Shape:", rag_reports_df.shape)

display(
    rag_reports_df.head(3)
)


RAG reports: 55
Shape: (55, 14)


,Machine_ID,Machine_Type,Cycle,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,maintenance_priority,maintenance_decision,alert_level,alert_required,retrieved_document_ids,retrieved_document_titles,rag_report
0,CNC_001,CNC_Mill,193,BEARING_WEAR,0.0002,47.87,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,"KB_GENERAL_001, KB_ACTUATOR_001, KB_BEARING_001",Predictive Maintenance — Warning and Critical ...,1. CURRENT ASSESSMENT \nThe CNC_001 machine (...
1,CNC_001,CNC_Mill,195,BEARING_WEAR,0.0002,49.58,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,"KB_GENERAL_001, KB_ACTUATOR_001, KB_BEARING_001",Predictive Maintenance — Warning and Critical ...,"1. CURRENT ASSESSMENT \nThe CNC_001 machine, ..."
2,CNC_001,CNC_Mill,221,BEARING_WEAR,0.0003,17.02,CRITICAL,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True,"KB_GENERAL_001, KB_BEARING_001, KB_ACTUATOR_001",Predictive Maintenance — Warning and Critical ...,"1. CURRENT ASSESSMENT \nThe CNC_001 machine, ..."


In [17]:
# ============================================
# STEP 17: VALIDATE RAG OUTPUT
# ============================================

if rag_reports_df.empty:
    raise RuntimeError(
        "No RAG reports were generated."
    )

missing_rag_reports = int(
    rag_reports_df["rag_report"]
    .isna()
    .sum()
)

empty_rag_reports = int(
    rag_reports_df["rag_report"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Missing RAG reports:", missing_rag_reports)
print("Empty RAG reports:", empty_rag_reports)

if missing_rag_reports == 0 and empty_rag_reports == 0:
    print("RAG output validation successful.")
else:
    print("WARNING: Some RAG reports are missing or empty.")


Missing RAG reports: 0
Empty RAG reports: 0
RAG output validation successful.


In [18]:
# ============================================
# STEP 18: DISPLAY A RAG REPORT
# ============================================

sample_rag = rag_reports_df.iloc[0]

print("============================================")
print("RAG MAINTENANCE INTELLIGENCE")
print("============================================")
print("Machine:", sample_rag["Machine_ID"])
print("Cycle:", sample_rag["Cycle"])
print("Fault:", sample_rag["Simulation_Fault"])
print("Decision:", sample_rag["maintenance_decision"])
print("Retrieved knowledge:")
print(sample_rag["retrieved_document_titles"])

print("\nRAG report:")
print(sample_rag["rag_report"])


RAG MAINTENANCE INTELLIGENCE
Machine: CNC_001
Cycle: 193
Fault: BEARING_WEAR
Decision: SCHEDULE INSPECTION
Retrieved knowledge:
Predictive Maintenance — Warning and Critical Conditions | Industrial Robot — Actuator Degradation | Rotating Equipment — Bearing Wear

RAG report:
1. CURRENT ASSESSMENT  
The CNC_001 machine (CNC_Mill) at cycle 193 is experiencing a simulated fault of BEARING_WEAR. The system has assigned a failure probability of 0.0002 (low likelihood currently) and a remaining useful life (RUL) of approximately 47.87 cycles. The overall status is WARNING, with a high maintenance priority and a decision to schedule an inspection. An alert is active and flagged as required.

2. HISTORICAL CONTEXT  
No prior maintenance history or machine events were found for CNC_001. This implies that the bearing wear condition is detected early in the machine's operation or that records are unavailable.

3. EVIDENCE  
The bearing wear fault is simulated and identified based on internal mode

In [19]:
# ============================================
# STEP 19: SAVE RAG OUTPUTS
# ============================================

rag_output_path = processed_dir / "rag_maintenance_reports.csv"

rag_reports_df.to_csv(
    rag_output_path,
    index=False
)

print("RAG maintenance reports saved:")
print(rag_output_path)


RAG maintenance reports saved:
/home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/rag_maintenance_reports.csv


In [20]:
# ============================================
# STEP 20: VERIFY RAG ARTIFACTS
# ============================================

if not rag_output_path.exists():
    raise FileNotFoundError(
        f"RAG output was not created: {rag_output_path}"
    )

saved_rag_df = pd.read_csv(rag_output_path)

with open(knowledge_path, "r", encoding="utf-8") as f:
    saved_knowledge = json.load(f)

print("RAG report file exists: True")
print("Knowledge base exists: True")
print("Saved RAG shape:", saved_rag_df.shape)
print("Knowledge documents:", len(saved_knowledge))

if len(saved_rag_df) == len(rag_reports_df):
    print("RAG report persistence verified.")


RAG report file exists: True
Knowledge base exists: True
Saved RAG shape: (55, 14)
Knowledge documents: 8
RAG report persistence verified.


In [21]:
# ============================================
# STEP 21: FINAL RAG SUMMARY
# ============================================

print("============================================")
print("RAG LAYER SUMMARY")
print("============================================")

print(
    "Actionable events processed:",
    len(rag_reports_df)
)

print(
    "Unique machines:",
    rag_reports_df["Machine_ID"].nunique()
)

print(
    "Knowledge documents:",
    len(knowledge_df)
)

print("\nMost frequently retrieved documents:")

retrieval_counts = (
    rag_reports_df["retrieved_document_ids"]
    .str.split(", ")
    .explode()
    .value_counts()
    .rename_axis("document_id")
    .reset_index(name="retrieval_count")
)

display(retrieval_counts)

print("\nFinal artifacts:")
print("Knowledge base:", knowledge_path)
print("RAG reports:", rag_output_path)

print("\nRAG layer completed successfully.")


RAG LAYER SUMMARY
Actionable events processed: 55
Unique machines: 7
Knowledge documents: 8

Most frequently retrieved documents:


,document_id,retrieval_count
0,KB_GENERAL_001,55
1,KB_BEARING_001,41
2,KB_ACTUATOR_001,31
3,KB_VIBRATION_001,21
4,KB_ELECTRICAL_001,9
5,KB_PRESSURE_001,8



Final artifacts:
Knowledge base: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/knowledge/maintenance_knowledge_base.json
RAG reports: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/rag_maintenance_reports.csv

RAG layer completed successfully.


# Final AI Architecture

After this notebook, the project contains the complete reasoning pipeline:

```text
                    IoT Simulation
                          ↓
                    Smart Twin
                          ↓
                 Failure + RUL Models
                          ↓
                    AI Agents
               ┌──────────┴──────────┐
               ↓                     ↓
        Maintenance Agent       Alert Agent
               └──────────┬──────────┘
                          ↓
                    Orchestrator
                          ↓
                  Actionable Events
                          ↓
                  ┌───────┴────────┐
                  ↓                ↓
               Memory             RAG
          machine history     maintenance knowledge
                  └───────┬────────┘
                          ↓
                         LLM
                          ↓
             Context-aware maintenance
                    intelligence
```

### The roles are intentionally separated

**ML models**
- predict failure probability
- predict RUL

**Agents**
- determine maintenance/alert actions

**Orchestrator**
- coordinates runtime decisions

**Memory**
- remembers machine history

**RAG**
- retrieves relevant maintenance knowledge

**LLM**
- synthesizes the supplied information into an operator-friendly explanation

The LLM is therefore an intelligence/explanation layer over the existing decision pipeline rather than the component that independently decides whether a machine should be repaired.
